# QUERY 5
Cantidad de transacciones del período [2022-09-01, 2022-09-05] con formato de pago
"Wire" o "ACH" cuyo monto convertido a USD sea menor a 1

In [1]:
import numpy as np
import pandas as pd
import requests

RUTA_DE_DATASETS = "../../datasets"
# NOMBRE_ARCHIVO = "LI-Small_Trans.csv"
NOMBRE_ARCHIVO = "transacciones_sample.csv"
SOLUCION = "q5_solucion.csv"

transacciones = pd.read_csv(f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO}")
print(f"Total filas cargadas: {len(transacciones)}")
transacciones.head()


Total filas cargadas: 100000


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/09 14:22,12,800194DE0,1291,8001FCF20,722.45,US Dollar,722.45,US Dollar,Credit Card,0
1,2022/09/02 06:18,13474,80E70A620,116473,811590780,31.27,US Dollar,31.27,US Dollar,Cheque,0
2,2022/09/01 07:35,70,10042B9C0,257741,8159D4B50,4392.31,Shekel,4392.31,Shekel,Cheque,0
3,2022/09/01 22:31,35779,8088038A0,35779,8088038A0,619.58,US Dollar,619.58,US Dollar,Reinvestment,0
4,2022/09/07 12:39,3,80020C290,17353,8055F9800,435361.79,Euro,435361.79,Euro,Cheque,0


In [2]:
# Filtro por período [2022-09-01, 2022-09-05]
# "2022/09/01" <= Timestamp <= "2022/09/05" (inclusive)
INICIO = "2022/09/01"
FIN    = "2022/09/06"  # "2022/09/06" para incluir todo el día 5/9

filtro_periodo = transacciones[
    (transacciones["Timestamp"] >= INICIO) &
    (transacciones["Timestamp"] < FIN)
]
print(f"Transacciones en el período: {len(filtro_periodo)}")
filtro_periodo.head()

Transacciones en el período: 54331


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
1,2022/09/02 06:18,13474,80E70A620,116473,811590780,31.27,US Dollar,31.27,US Dollar,Cheque,0
2,2022/09/01 07:35,70,10042B9C0,257741,8159D4B50,4392.31,Shekel,4392.31,Shekel,Cheque,0
3,2022/09/01 22:31,35779,8088038A0,35779,8088038A0,619.58,US Dollar,619.58,US Dollar,Reinvestment,0
8,2022/09/02 22:11,14003,801A7A460,7272,808606A90,1510.95,Euro,1510.95,Euro,Cheque,0
9,2022/09/02 17:53,21444,800EF0540,2514,801DB6ED0,870.56,Euro,870.56,Euro,Cheque,0


In [3]:
# Filtro por formato de pago: Wire o ACH
FORMATOS = ["Wire", "ACH"]

filtro_formato = filtro_periodo[filtro_periodo["Payment Format"].isin(FORMATOS)]
print(f"Transacciones Wire/ACH en el período: {len(filtro_formato)}")
filtro_formato.head()


Transacciones Wire/ACH en el período: 7620


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
22,2022/09/01 22:23,212,80F0D5AC0,141465,810A56D90,1816.04,Australian Dollar,1816.04,Australian Dollar,ACH,0
38,2022/09/01 01:16,17059,809C350C0,130803,80D1499B0,135.68,Euro,135.68,Euro,ACH,0
45,2022/09/03 22:01,20,8011E85D0,20,8011E85D0,150397.03,Yuan,22455.36,US Dollar,ACH,0
47,2022/09/02 16:48,11,8001048F0,12,800106DA0,52267.41,US Dollar,52267.41,US Dollar,Wire,0
81,2022/09/05 17:24,25526,806918720,124049,80CE81C60,1379.55,US Dollar,1379.55,US Dollar,Wire,0


In [4]:
# Descarga de cotizaciones desde API Frankfurter (base EUR)
# y forward-fill para días sin mercado (fines de semana / feriados)
from datetime import date, timedelta

url = "https://api.frankfurter.app/2022-09-01..2022-09-05"
resp = requests.get(url, timeout=10)
resp.raise_for_status()
cotizaciones_raw = resp.json()["rates"]   # solo días hábiles
print("Fechas con cotización de la API:", list(cotizaciones_raw.keys()))

# Expandir el rango completo usando la última cotización conocida
cotizaciones = {}
last_rates = None
dia = date(2022, 9, 1)
fin = date(2022, 9, 5)
while dia <= fin:
    key = dia.strftime("%Y-%m-%d")
    if key in cotizaciones_raw:
        last_rates = cotizaciones_raw[key]
    if last_rates is not None:
        cotizaciones[key] = last_rates
    dia += timedelta(days=1)

print("Fechas disponibles tras forward-fill:", list(cotizaciones.keys()))


Fechas con cotización de la API: ['2022-09-01', '2022-09-02', '2022-09-05']
Fechas disponibles tras forward-fill: ['2022-09-01', '2022-09-02', '2022-09-03', '2022-09-04', '2022-09-05']


In [5]:
# Conversión de monto recibido a USD (misma lógica que el worker distribuido)
CURRENCY_MAP = {
    "US Dollar":         "USD",
    "Euro":              "EUR",
    "UK Pound":          "GBP",
    "Yen":               "JPY",
    "Australian Dollar": "AUD",
    "Bitcoin":           "BTC",
    "Brazil Real":       "BRL",
    "Canadian Dollar":   "CAD",
    "Mexican Peso":      "MXN",
    "Ruble":             "RUB",
    "Rupee":             "INR",
    "Saudi Riyal":       "SAR",
    "Shekel":            "ILS",
    "Swiss Franc":       "CHF",
    "Yuan":              "CNY",
}

def convertir_a_usd(row):
    iso = CURRENCY_MAP.get(row["Receiving Currency"])
    if not iso:
        return None
    if iso == "USD":
        return float(row["Amount Received"])

    fecha = row["Timestamp"].split(" ")[0].replace("/", "-")
    rates = cotizaciones.get(fecha)  # ya tiene forward-fill para fines de semana
    if not rates:
        return None

    rate_usd = rates.get("USD")
    if not rate_usd:
        return None

    monto = float(row["Amount Received"])
    if iso == "EUR":
        return monto * rate_usd

    # Triangulación: origen → EUR → USD
    rate_origen = rates.get(iso)
    if not rate_origen:
        return None
    return (monto / rate_origen) * rate_usd

filtro_formato = filtro_formato.copy()
filtro_formato["amount_usd"] = filtro_formato.apply(convertir_a_usd, axis=1)

# Descartamos filas sin cotización (divisa no mapeada) y filtramos monto < 1 USD
menores_1_usd = filtro_formato.dropna(subset=["amount_usd"])
menores_1_usd = menores_1_usd[menores_1_usd["amount_usd"] < 1.0]
print(f"Transacciones < 1 USD: {len(menores_1_usd)}")
menores_1_usd[["Timestamp", "Receiving Currency", "Amount Received", "amount_usd"]].head()


Transacciones < 1 USD: 114


,Timestamp,Receiving Currency,Amount Received,amount_usd
1013,2022/09/01 00:10,Yuan,0.05,0.007247
1032,2022/09/01 17:06,Shekel,0.09,0.026761
1466,2022/09/01 14:29,Rupee,0.47,0.005905
1536,2022/09/05 02:44,Euro,0.13,0.128960
3898,2022/09/01 12:19,Mexican Peso,0.13,0.006440


In [6]:
# Resultado final: cantidad de transacciones
count = len(menores_1_usd)
resultado = pd.DataFrame({"count": [count]})
print(f"Q5 resultado: {count} transacciones")

resultado.to_csv(SOLUCION, index=False)
print(f"Guardado en {SOLUCION}")


Q5 resultado: 114 transacciones
Guardado en q5_solucion.csv
